# Module 6 — Advanced RAG & Vibe Coding

**Duration:** ~75 minutes  
**Goal:** Apply advanced retrieval techniques and measure their impact on RAGAS metrics.

---

## What's in this module

| Technique | Problem it solves | Expected metric improvement |
|-----------|------------------|----------------------------|
| **Query expansion** | Sparse query misses synonyms | ↑ Context Recall |
| **HyDE** (Hypothetical Document Embeddings) | Query and doc embeddings live in different spaces | ↑ Context Precision |
| **Re-ranking** | Initial retrieval returns noise | ↑ Context Precision |
| **Hybrid search** | Semantic search misses exact terms | ↑ Context Recall |

---

## Advanced RAG Architecture

```
Question
   │
   ├─ Query expansion  → [Q, Q+synonym, Q+context]
   │       │
   ├─ HyDE            → generate hypothetical answer → embed → retrieve
   │       │
   ├─ Retrieve        → top-20 candidates (over-retrieve)
   │       │
   ├─ Re-rank         → cross-encoder scores → top-5
   │       │
   └─ Generate        → answer with top-5 re-ranked chunks
```

In [ ]:
import json
import os

from openai import OpenAI
import chromadb
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder

load_dotenv("../.env")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path="../chroma_db")
collection = chroma_client.get_or_create_collection(
    name="workshop_rag", metadata={"hnsw:space": "cosine"}
)
llm_client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.ai/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)
FAST_MODEL = os.getenv("NETLIGHT_MODEL_FAST", "claude-haiku-4-5")

# Base retrieve function
def retrieve(question: str, top_k: int = 5) -> list[dict]:
    q_emb = embed_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[q_emb], n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    return [
        {"text": t, "source": m["source"], "similarity": 1 - d}
        for t, m, d in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]

print(f"Collection: {collection.count()} docs")

---
## Technique 1 — Query Expansion

**Problem:** A user asks "What does an actuary do?" — but the document says  
"Actuarial professionals assess risk using statistical methods".  
The embeddings may not capture this synonym gap.

**Solution:** Ask Claude to generate 2–3 alternative phrasings of the query before retrieval.

### Exercise 5.1 — Implement multi-query retrieval

> **Task:** Use Claude to generate N alternative queries for a given question.  
> Retrieve chunks for each query, then deduplicate and return the union.
>
> **Hints:**
> - Prompt Claude to return a JSON list of alternative queries
> - Use a `dict` keyed by chunk text for deduplication
> - Keep the original query too — always include its results

In [ ]:
def expand_query(question: str, n: int = 3) -> list[str]:
    """
    Use Claude to generate N alternative phrasings of a question.
    Returns a list of queries including the original.
    """
    prompt = f"""Generate {n} alternative phrasings of this question for semantic search.
Return ONLY a JSON array of strings, no explanation.

Question: {question}"""

    response = llm_client.chat.completions.create(
        model=FAST_MODEL, max_tokens=256,
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.choices[0].message.content.strip()

    # TODO: parse JSON, handle errors, prepend original question
    try:
        alternatives = json.loads(text)
        return [question] + alternatives[:n]
    except json.JSONDecodeError:
        return [question]


# Test
q = "What does DKV hospitalisation insurance cover?"
expanded = expand_query(q)
print(f"Original: {q}")
print("Expanded queries:")
for eq in expanded:
    print(f"  - {eq}")

In [ ]:
def retrieve_multi_query(question: str, top_k_per_query: int = 3) -> list[dict]:
    """
    Expand the query and retrieve chunks for each variant.
    Deduplicate by chunk text and return unique chunks.
    """
    queries = expand_query(question, n=2)
    seen_texts = {}

    for q in queries:
        for chunk in retrieve(q, top_k=top_k_per_query):
            # TODO: add to seen_texts dict, deduplicating by chunk text
            pass

    # TODO: return deduplicated list, sorted by similarity
    return []


# Compare baseline vs multi-query
test_q = "What does DKV hospitalisation insurance cover?"

baseline_chunks = retrieve(test_q, top_k=5)
multiquery_chunks = retrieve_multi_query(test_q, top_k_per_query=3)

print(f"Baseline retrieval: {len(baseline_chunks)} chunks")
print(f"Multi-query retrieval: {len(multiquery_chunks)} unique chunks")
print(f"Sources (baseline): {list(set(c['source'] for c in baseline_chunks))}")
print(f"Sources (multi-query): {list(set(c['source'] for c in multiquery_chunks))}")

---
## Technique 2 — HyDE (Hypothetical Document Embeddings)

**Problem:** A user's question embedding (`"What is reinsurance?"`) lives in a different  
vector space than the document chunk embedding (`"Reinsurance is insurance purchased by..."`).

**HyDE idea:** Instead of embedding the question, ask Claude to generate a *hypothetical answer*  
and embed that instead. Documents and the hypothetical answer are now in the same space.

```
Without HyDE: embed("What is reinsurance?")          → retrieve
With HyDE:    embed(Claude("Write a short text about reinsurance")) → retrieve
```

### Exercise 5.2 — Implement HyDE retrieval

In [ ]:
def hyde_retrieve(question: str, top_k: int = 5) -> list[dict]:
    """
    1. Ask Claude to generate a hypothetical document that would answer the question
    2. Embed that hypothetical document
    3. Use it as the query embedding for retrieval
    """
    # Step 1: Generate hypothetical document
    hyde_prompt = f"""Write a short factual paragraph (3-5 sentences) that would directly answer this question.
Do not say you are answering a question — just write the paragraph.

Question: {question}"""

    response = llm_client.chat.completions.create(
        model=FAST_MODEL, max_tokens=256,
        messages=[{"role": "user", "content": hyde_prompt}]
    )
    hypothetical_doc = response.choices[0].message.content

    # Step 2 & 3: TODO — embed the hypothetical doc and use it to query ChromaDB
    # hyde_embedding = ...
    # results = collection.query(...)

    print(f"Hypothetical doc: {hypothetical_doc[:200]}...")
    return []  # TODO: return actual chunks


# Compare
test_q = "What physiotherapy sessions does DKV reimburse after hospitalisation?"

baseline_chunks = retrieve(test_q, top_k=5)
hyde_chunks = hyde_retrieve(test_q, top_k=5)

print(f"\nBaseline sources: {[c['source'] for c in baseline_chunks]}")
print(f"HyDE sources: {[c['source'] for c in hyde_chunks]}")

---
## Technique 3 — Re-ranking with a Cross-Encoder

**Problem:** The bi-encoder (sentence-transformers) retrieves quickly but imprecisely.  
A cross-encoder computes a direct relevance score between the query and each chunk — much more accurate, but slower.

**Strategy:** Over-retrieve with the fast bi-encoder (top-20), then re-rank with the cross-encoder and keep top-5.

```
Bi-encoder retrieval → [c1, c2, ..., c20] (fast, approximate)
Cross-encoder scoring → relevance(query, ci) for each chunk
Re-ranked top-5 → much higher precision
```

### Exercise 5.3 — Implement re-ranking

In [ ]:
# Load cross-encoder (downloads ~65 MB on first run)
print("Loading cross-encoder (first run ~65 MB)...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded")

In [ ]:
def retrieve_with_rerank(question: str, initial_k: int = 20, final_k: int = 5) -> list[dict]:
    """
    1. Over-retrieve initial_k chunks with bi-encoder
    2. Re-rank with cross-encoder
    3. Return top final_k
    """
    # Step 1: Initial retrieval
    candidates = retrieve(question, top_k=initial_k)

    # Step 2: TODO — score each candidate with the cross-encoder
    # cross_encoder.predict([(query, text), ...]) returns scores
    pairs = [(question, c["text"]) for c in candidates]
    # scores = cross_encoder.predict(pairs)

    # Step 3: TODO — sort by cross-encoder score and return top final_k
    # ...

    return candidates[:final_k]  # placeholder — replace with re-ranked results


# Compare bi-encoder vs re-ranked
test_q = "What are the reimbursement limits for dental treatments under DKV?"

print("=== Bi-encoder only (top 5) ===")
bi_chunks = retrieve(test_q, top_k=5)
for i, c in enumerate(bi_chunks, 1):
    print(f"  [{i}] sim={c['similarity']:.3f} | {c['source']} | {c['text'][:80]}...")

print("\n=== Re-ranked (top 5 from 20) ===")
reranked_chunks = retrieve_with_rerank(test_q, initial_k=20, final_k=5)
for i, c in enumerate(reranked_chunks, 1):
    score = c.get("rerank_score", c["similarity"])
    print(f"  [{i}] score={score:.3f} | {c['source']} | {c['text'][:80]}...")

---
## Putting it together — Advanced RAG Pipeline

### Exercise 5.4 — Build an `AdvancedRAGPipeline`

> **Task:** Combine the techniques into a configurable pipeline with flags to enable/disable each technique.
>
> ```python
> pipeline = AdvancedRAGPipeline(
>     use_query_expansion=True,
>     use_hyde=False,
>     use_reranking=True,
> )
> ```
>
> **Goal:** Run this against RAGAS and compare scores with the baseline from Module 4.

In [ ]:
SYSTEM_PROMPT = """\
You are a knowledgeable insurance and technology assistant.
Answer ONLY using information from the numbered context chunks provided.
Cite sources using the chunk number, e.g. "According to [1]..."
If the context is insufficient, say so. Never fabricate facts."""


def build_prompt(question: str, chunks: list[dict]) -> str:
    lines = []
    for i, chunk in enumerate(chunks, 1):
        lines.append(f"[{i}] Source: {chunk['source']}")
        lines.append(chunk["text"])
        lines.append("")
    return f"Retrieved Context:\n{''.join(lines)}\nQuestion: {question}\n\nAnswer:"


class AdvancedRAGPipeline:
    def __init__(
        self,
        collection,
        embed_model,
        llm_client,
        model: str = FAST_MODEL,
        top_k: int = 5,
        use_query_expansion: bool = False,
        use_hyde: bool = False,
        use_reranking: bool = False,
        rerank_initial_k: int = 20,
    ):
        self.collection = collection
        self.embed_model = embed_model
        self.llm_client = llm_client
        self.model = model
        self.top_k = top_k
        self.use_query_expansion = use_query_expansion
        self.use_hyde = use_hyde
        self.use_reranking = use_reranking
        self.rerank_initial_k = rerank_initial_k
        if use_reranking:
            self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def ask(self, question: str) -> dict:
        # TODO: implement with flags for each technique
        # 1. Optionally expand query
        # 2. Optionally use HyDE
        # 3. Retrieve
        # 4. Optionally re-rank
        # 5. Generate
        pass


# Test different configurations
configs = {
    "baseline": {"use_query_expansion": False, "use_hyde": False, "use_reranking": False},
    "+expansion": {"use_query_expansion": True, "use_hyde": False, "use_reranking": False},
    "+reranking": {"use_query_expansion": False, "use_hyde": False, "use_reranking": True},
    "all": {"use_query_expansion": True, "use_hyde": False, "use_reranking": True},
}

test_question = "What are the waiting periods and exclusions for DKV hospitalisation insurance?"

print(f"Question: {test_question}\n")
for config_name, config_kwargs in configs.items():
    pipeline = AdvancedRAGPipeline(
        collection=collection, embed_model=embed_model, llm_client=llm_client,
        top_k=5, **config_kwargs
    )
    if pipeline.ask:
        result = pipeline.ask(test_question)
        if result:
            print(f"[{config_name}]")
            print(f"  Sources: {set(result.get('sources', []))}")
            print(f"  Answer: {result.get('answer', '')[:150]}...\n")

---
## Vibe Coding Challenge

### The Group Challenge (30 minutes)

**Goal:** Improve the RAGAS scores from Module 4 by at least 5% on two metrics.

You have the evaluation framework. Use AI tools (Claude, GitHub Copilot, etc.) to implement your ideas.

**Ideas to try:**

1. **Sentence-based chunking** — Does it improve context precision over word-based?

2. **Larger `top_k` + re-ranking** — Retrieve 15, re-rank to 5. Does precision improve?

3. **Better system prompt** — What prompt phrasing maximises faithfulness?

4. **`claude-sonnet-4-6` vs `claude-haiku-4-5`** — How much does model quality matter?

5. **Metadata filtering** — Can source-filtered retrieval improve context precision?

6. **Hybrid retrieval** — Combine semantic + BM25 keyword search (try `rank_bm25`).

7. **Bigger embedding model** — Swap `all-MiniLM-L6-v2` for `all-mpnet-base-v2`. Does it help?

**Vibe coding workflow:**
1. Describe the change to your AI assistant
2. Get the implementation
3. Run the RAGAS evaluation
4. Record the delta in the scoreboard below

In [ ]:
# Scoreboard — fill in as you experiment
scoreboard = [
    {"config": "Baseline (Module 4)", "faithfulness": None, "answer_relevancy": None,
     "context_precision": None, "context_recall": None},
    # {"config": "Your experiment name", "faithfulness": 0.85, ...},
]

# Visualise scoreboard
if len(scoreboard) > 1:
    import pandas as pd
    metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    df = pd.DataFrame(scoreboard).set_index("config")
    df[metrics].plot(kind="bar", figsize=(12, 5), ylim=(0, 1))
    plt.title("Configuration Comparison")
    plt.ylabel("Score")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Add your experiment scores to the scoreboard above!")

---
## Workshop Retrospective

### What we learned

```
Day 1 — Build
  ✓ RAG architecture end-to-end
  ✓ Chunking strategies and their trade-offs
  ✓ Embeddings with sentence-transformers (local, free)
  ✓ ChromaDB vector store
  ✓ RAGPipeline class with configurable prompt engineering

Day 2 — Evaluate & Improve
  ✓ RAGAS: Faithfulness, Answer Relevancy, Context Precision/Recall
  ✓ Failure mode diagnosis
  ✓ A/B testing framework
  ✓ Query expansion, HyDE, re-ranking
  ✓ Measured impact of each technique
```

### What's next

| Topic | Where to look |
|-------|---------------|
| Agentic RAG (tool use) | Anthropic docs → Tool use |
| Graph RAG | Microsoft GraphRAG repo |
| Production deployment | FastAPI + Docker + cloud vector DB |
| LangChain/LlamaIndex | If you need pre-built pipeline abstractions |
| Tracing & monitoring | Langfuse, Arize Phoenix |
| Fine-tuned embeddings | Domain-specific embedding models |

### The key insight

> **RAG is a system, not a model.**  
> Every component — chunking, embedding, retrieval, prompting, generation — has a measurable effect on quality.  
> The evaluation framework is not optional: it's what turns RAG from a demo into a product.